In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
# %pip install tiktoken
import tiktoken

In [2]:
#Building the Dataset
f = open('tiny-shakespeare.txt')
text = f.read()

print("Length of the Dataset:", len(text), "\n")
print(text[:100]) #first 100 characters

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'\nVocabulary Size: {vocab_size}')
print('-'.join(chars))

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i: ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ary: ''.join([itos[i] for i in ary])
raw_text = "ben Onur"
print(f'\nRaw Text:  {raw_text}')
token_list = encode(raw_text)
print("Token List:", token_list)
decoded = decode(token_list)
print("decoded:", decoded)

# Compare with OpenAI byte-pair encoding (BPE)
# Tiktoken shows that there is a trade-off between the length of the encoding and the amount of tokens.
# We can have short sequences of tokens with very large vocabulary, or we can just as well have long sequences of tokens with a small vocabulary.
# The BPE approach is widely used for NLP tasks
enc = tiktoken.get_encoding('gpt2')

token_list_BPE = enc.encode(raw_text)
print("token_list_BPE:", token_list_BPE) # BPE returns fewer tokens than the character encoding
print(enc.decode(enc.encode(raw_text)))

print(enc.n_vocab) # total amount of tokens in the vocabulary


Length of the Dataset: 1115394 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

Vocabulary Size: 65

- -!-$-&-'-,---.-3-:-;-?-A-B-C-D-E-F-G-H-I-J-K-L-M-N-O-P-Q-R-S-T-U-V-W-X-Y-Z-a-b-c-d-e-f-g-h-i-j-k-l-m-n-o-p-q-r-s-t-u-v-w-x-y-z

Raw Text:  ben Onur
Token List: [40, 43, 52, 1, 27, 52, 59, 56]
decoded: ben Onur
token_list_BPE: [11722, 1550, 333]
ben Onur
50257


In [ ]:
# Encode the text into a tensor of integers
encoded_data = encode(text)
data = torch.tensor(encoded_data, dtype = torch.long)
print(f'Total size: {data.shape} elements of type {data.dtype}')
print('First 10 tokens from the dataset:', data[:10])

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

def build_dataset(encoded_data, block_size, batch_size):
    ix = torch.randint(
        0,
        len(encoded_data) - block_size,
        (batch_size,)
    )

    X = torch.stack([
        encoded_data[i:i + block_size]
        for i in ix
    ])

    Y = torch.stack([
        encoded_data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return X, Y
        


Total size: torch.Size([1115394]) elements of type torch.int64
First 10 tokens from the dataset: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [17]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
stacked = torch.stack([a, b], dim=0)
print(stacked)
print(stacked.shape)

stacked2 = torch.stack([stacked, stacked], dim=2)
print(stacked2)
print(stacked2.shape)

tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])
tensor([[[1, 1],
         [2, 2],
         [3, 3]],

        [[4, 4],
         [5, 5],
         [6, 6]]])
torch.Size([2, 3, 2])
